In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

print("\n--- NGÀY 74: DẠY AI TẬP NÓI (NEXT-WORD PREDICTION) ---")

#1. Chuẩn bị dữ liệu
van_ban = [
    "Tôi rất thích học trí tuệ nhân tạo", 
    "Tôi rất thích lập trình Python",
    "Trí tuệ nhân tạo cực kỳ thú vị"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(van_ban)

#Cộng thêm 1 cho padding
tong_so_tu = len(tokenizer.word_index)+1
print(f"Từ điển của 'Tiểu GPT' có {tong_so_tu - 1} từ vựng.")

#3. Chuẩn bị đề thi (đọc đến đâu đoán từ đến đó)
input_sequences = []
for line in van_ban:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

#Kéo dãn cho bằng nhau
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
X = input_sequences[:, :-1]
y_label = input_sequences[:, -1]

# CỰC KỲ QUAN TRỌNG: Biến y thành mảng One-hot encoding (xác suất cho toàn bộ từ điển)
y = to_categorical(y_label, num_classes=tong_so_tu)

#4. Xây dựng mô hình
model_gpt = Sequential()
model_gpt.add(Embedding(input_dim=tong_so_tu, output_dim=10, input_length=max_sequence_len-1))
model_gpt.add(LSTM(64))
# Đầu ra bây giờ không phải là 1 nơ-ron nữa, mà bằng đúng TỔNG SỐ TỪ trong từ điển
# Hàm softmax sẽ tính xem từ nào có xác suất cao nhất sẽ được bật lên!
model_gpt.add(Dense(tong_so_tu, activation='softmax'))

model_gpt.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

#5. Huấn luyện mô hình
print("Đang dạy AI tập nói... (Cày 200 vòng cho nhớ thật kỹ)")
# Bỏ qua tập Validation vì dữ liệu quá nhỏ, ép nó học thuộc lòng luôn!
model_gpt.fit(
    X,
    y,
    epochs=200,
    verbose=1
)
print("Huấn luyện xong!")

#6. Dự đoán
cau_mo_dau = "Tôi rất yêu"
# Bước này hơi giống vòng lặp của ChatGPT:
# Đưa "Tôi rất thích" vào -> AI nhả chữ "học"
# Dịch mã số "học" thành chữ, rồi in ra.
print(f"\nCon người gõ: '{cau_mo_dau}'")
print("Tiểu GPT tự động viết tiếp:")

#Chuẩn bị số hóa câu mở đầu
token_list = tokenizer.texts_to_sequences([cau_mo_dau])[0]
token_list = pad_sequences([token_list],maxlen=max_sequence_len-1, padding='pre')

#Dự đoán từ tiếp theo
du_doan = model_gpt.predict(token_list, verbose=1)
tu_tiep_theo_ID = np.argmax(du_doan)

#Tim lai tu goc
tu_tiep_theo_chu = ""
for word, index in tokenizer.word_index.items():
    if index == tu_tiep_theo_ID:
        tu_tiep_theo_chu = word
        break

print(f"=> {cau_mo_dau} ____{tu_tiep_theo_chu}____")

print(f"Độ tự tin: { np.max(du_doan)*100:.2f}%")

In [13]:
print("\n--- NGÀY 75: VÒNG LẶP SÁNG TẠO VĂN BẢN ---")

#1.Hàm sinnh văn bản
def sinh_van_ban(cau_mo_dau, do_dai):
    cau_hien_tai = cau_mo_dau
    for _ in range(do_dai):
        #Số hóa câu hiện tại
        token_list = tokenizer.texts_to_sequences([cau_hien_tai])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        #Dự đoán từ tiếp theo
        du_doan = model_gpt.predict(token_list, verbose=0)
        tu_tiep_theo_id = np.argmax(du_doan, axis=-1)[0]

        #Dịch mã sang chữ
        tu_tiep_theo_chu=""
        for tu,index in tokenizer.word_index.items():
            if index == tu_tiep_theo_id:
                tu_tiep_theo_chu = tu
                break
        cau_hien_tai+= " " +tu_tiep_theo_chu
    return cau_hien_tai
    
#2. Thử sinh văn bản
cau_kieu_1 = "Trí tuệ"
cau_kieu_2 = "Tôi"

print(f"\n[Gợi ý]: '{cau_kieu_1}'")
print(f"=> AI viết tiếp 4 từ: {sinh_van_ban(cau_kieu_1, 6)}")
print(f"\n[Gợi ý]: '{cau_kieu_2}'")
print(f"=> AI viết tiếp 4 từ: {sinh_van_ban(cau_kieu_2, 5)}")



--- NGÀY 75: VÒNG LẶP SÁNG TẠO VĂN BẢN ---

[Gợi ý]: 'Trí tuệ'
=> AI viết tiếp 4 từ: Trí tuệ nhân tạo cực kỳ thú vị

[Gợi ý]: 'Tôi'
=> AI viết tiếp 4 từ: Tôi rất thích lập trình python
